# grid_grid_fusion 方法说明

将源格点场重采样到目标水平网格。包内两类入口：

| 系列 | 插件 | 典型用途 |
| --- | --- | --- |
| 等距经纬插值（InterpGg*） | `InterpGgLinear` / `Nearest` / `Cubic` | 等距经纬、融合/背景对齐；越界用 `outer_value` |
| LandSea | `InterpGgLandSea`（及 `InterpGgLandSea2`） | RGI / `*-2` 重网格；可选海陆掩码 |

在包根或本 `nbs/` 目录打开本 notebook 即可运行示例（依赖 `meteva_base`、`resource/data`）。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import meteva_base as meb
import numpy as np

PACKAGE_ROOT = Path.cwd().resolve()
if PACKAGE_ROOT.name == "nbs":
    PACKAGE_ROOT = PACKAGE_ROOT.parent
DATA_DIR = PACKAGE_ROOT / "resource" / "data"

src_s, root_s = str(PACKAGE_ROOT / "src"), str(PACKAGE_ROOT)
for p in (src_s, root_s):
    if p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, src_s)
sys.path.insert(0, root_s)

from interp_gg_cubic_plugin import InterpGgCubic
from interp_gg_landsea_plugin import InterpGgLandSea
from interp_gg_linear_plugin import InterpGgLinear
from interp_gg_nearest_plugin import InterpGgNearest

print("PACKAGE_ROOT =", PACKAGE_ROOT)
print("DATA_DIR exists =", DATA_DIR.is_dir())

d:\workspace\test\.venv\Lib\site-packages\meteva_base\basicdata\const.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


PACKAGE_ROOT = D:\workspace\test\00temp\grid_grid_fusion
DATA_DIR exists = True


## 1. 等距经纬插值（InterpGg*）：原理

接口统一为 `process(grd, grid)`：`grd` 为源六维场，`grid` 为目标 `meb.grid`（等距经纬）。

| 方法 | 原理 |
| --- | --- |
| **InterpGgLinear** | 目标点映射到源索引分数，取四邻点双线性加权 |
| **InterpGgNearest** | 索引四舍五入到最近源格点赋值 |
| **InterpGgCubic** | 对邻域 `p,q ∈ {-1,0,1,2}` 用三次核 `cubic_f` 加权 |

共同约定：

- 源经度跨满 360°（`iscycle`）时经度取模；**纬度越界仍须** `outer_value`。
- 非循环场经纬越界：先插重叠区，再 `expand_to_contain_another_grid` 填外区。
- `member` / `level` / `time` / `dtime` 继承源场；水平网格取自目标 `grid`。
- `attrs` 深拷贝自源场。

## 2. 等距经纬插值（InterpGg*）：参数

| 参数 | 位置 | 说明 |
| --- | --- | --- |
| `used_coords` | 构造 / `process` | 当前仅支持 `"xy"` |
| `outer_value` | 构造 / `process` | 目标超出源域时的填充值；越界且未给则返回 `None` |
| `grd` | `process` | 源格点场 |
| `grid` | `process` | 目标网格信息（`glon` / `glat` 等） |

CLI（在包目录）：`python -m cli --module linear|nearest|cubic ...`，缺省读 `resource/conf.ini`。

## 3. InterpGg 使用示例

In [2]:
src = meb.read_griddata_from_nc(str(DATA_DIR / "2026100900.024.nc"))
# 目标：略加密、落在源域内
tgt = meb.basicdata.grid([80.0, 100.0, 0.05], [30.0, 40.0, 0.05])

out_lin = InterpGgLinear(outer_value=np.nan).process(src, tgt)
out_near = InterpGgNearest(outer_value=np.nan).process(src, tgt)
out_cub = InterpGgCubic(outer_value=np.nan).process(src, tgt)

for name, da in ("Linear", out_lin), ("Nearest", out_near), ("Cubic", out_cub):
    print(name, dict(da.sizes), "minmax",
          float(np.nanmin(da.values)), float(np.nanmax(da.values)))

Linear {'member': 1, 'level': 1, 'time': 1, 'dtime': 1, 'lat': 201, 'lon': 401} minmax -27.700000762939453 13.399999618530273
Nearest {'member': 1, 'level': 1, 'time': 1, 'dtime': 1, 'lat': 201, 'lon': 401} minmax -27.700000762939453 13.399999618530273
Cubic {'member': 1, 'level': 1, 'time': 1, 'dtime': 1, 'lat': 201, 'lon': 401} minmax -27.700000762939453 13.399999618530273


## 4. LandSea：原理

`InterpGgLandSea` 为统一入口，按 `regrid_mode` 分派：

| `regrid_mode` | 路径 | 掩码 |
| --- | --- | --- |
| `bilinear` / `nearest` | `RegularGridInterpolator`（scipy） | 否 |
| `nearest-with-mask` | 先最近邻，再 `AdjustLandSeaPoints` 订正 | 要源+目标掩码 |
| `bilinear-2` / `nearest-2` | `InterpGgLandSea2` 自研核 | 否 |
| `*-with-mask-2` | `InterpGgLandSea2`：插值时按海陆改邻点/权重 | 要源+目标掩码 |

**掩码约定**：`land_binary_mask`，陆=1、海=0。  
- 源掩码：构造参数 `landmask`。  
- 目标掩码：`process` 的第二参 `target_grid`（掩码模式下该场本身即目标掩码，并提供目标网格）。

`nearest-with-mask`：插值后再用邻域（`landmask_vicinity`，默认 25 km）内同类型最近点替换误取海/陆的点。  
`*-with-mask-2`：在四邻点阶段就按海陆筛选，不是纯后处理。

## 5. LandSea：参数

### `InterpGgLandSea`

| 参数 | 说明 |
| --- | --- |
| `regrid_mode` | 见上表；默认 `bilinear` |
| `extrapolation_mode` | `nan`（域外填 NaN，默认）/ `extrapolate` / `error`（历史别名 mask/nanmask → nan） |
| `landmask` | 源侧海陆掩码；`*-with-mask*` 必需 |
| `landmask_vicinity` | 同类型搜索半径（米），默认 `25000` |
| `process(data, target_grid, regridded_title=None)` | `data` 源场；`target_grid` 目标网格模板（掩码模式为目标掩码场） |

### `InterpGgLandSea2`（一般经 `InterpGgLandSea` 调用）

| 参数 | 说明 |
| --- | --- |
| `regrid_mode` | `bilinear-2` / `nearest-2` / `*-with-mask-2` |
| `vicinity_radius` | 同 `landmask_vicinity` |
| `process(data_in, data_in_mask, data_out_mask)` | 无掩码时 `data_in_mask` 可为 `None` |

CLI：`python -m cli.tran_regrid --input-path ... --target-grid-path ... [--land-sea-mask-path ...] --regrid-mode ...`

## 6. LandSea 使用示例（无掩码）

以预报场为源、HGT 子集为目标网格（仅取空间坐标）。

In [3]:
src = meb.read_griddata_from_nc(str(DATA_DIR / "2026100900.024.nc"))
hgt = meb.read_griddata_from_nc(str(DATA_DIR / "HGT_M_1KM_CHINA.nc"))
hgt_tgt = hgt.sel(lon=slice(110.0, 112.0), lat=slice(35.0, 37.0))

g_hgt = meb.get_grid_of_data(hgt_tgt)
g0 = meb.get_grid_of_data(src)
tmpl = meb.grid_data(
    meb.grid(g_hgt.glon, g_hgt.glat, g0.gtime, g0.dtimes, g0.levels, g0.members)
)
tmpl[:] = 0

out_bi = InterpGgLandSea(regrid_mode="bilinear", extrapolation_mode="nan").process(src, tmpl)
out_nn = InterpGgLandSea(regrid_mode="nearest", extrapolation_mode="nan").process(src, tmpl)
out_bi2 = InterpGgLandSea(regrid_mode="bilinear-2", extrapolation_mode="nan").process(src, tmpl)

print("bilinear ", out_bi.shape, float(np.nanmin(out_bi)), float(np.nanmax(out_bi)))
print("nearest  ", out_nn.shape, float(np.nanmin(out_nn)), float(np.nanmax(out_nn)))
print("bilinear-2", out_bi2.shape, float(np.nanmin(out_bi2)), float(np.nanmax(out_bi2)))

bilinear  (1, 1, 1, 1, 201, 201) 9.600000381469727 18.700000762939453
nearest   (1, 1, 1, 1, 201, 201) 9.600000381469727 18.700000762939453
bilinear-2 (1, 1, 1, 1, 201, 201) 9.600000381469727 18.700000762939453


## 7. 选型建议

- **业务融合 / 背景对齐（meb 等距经纬）**：优先 `InterpGgLinear` / `Nearest` / `Cubic`，越界用 `outer_value`。
- **与 IMPROVER 海陆模式对齐**：`InterpGgLandSea`；需掩码时准备源、目标两份 `land_binary_mask`。
- 等距经纬、域内：InterpGgLinear 与 LandSea `bilinear` 数值接近；最近邻在格心附近可能有格点级差。
- 更多海陆细节见 `docs/grid_grid_fusion.md`；数值对比见 `nbs/cmp_gg_vs_regrid.ipynb`。